# A_02 · Python NumPy Tutorial

<table>
<tr><td><b>Student</b></td><td>Nabonita Das</td></tr>
<tr><td><b>Email</b></td><td>nabonita@umich.edu</td></tr>
<tr><td><b>Student ID</b></td><td>35226076 (UMich ID)</td></tr>
<tr><td><b>Course</b></td><td>ECE 5831-001 / ECE 5831-002 (Fall 2026)</td></tr>
<tr><td><b>Assignment</b></td><td>HW#2 · <code>ece-5831-2026-assignments/A_02</code></td></tr>
<tr><td><b>GSI</b></td><td>Elahe Delavari (GitHub: <code>ElaheDlv</code>)</td></tr>
<tr><td><b>Due</b></td><td>Tuesday, September 29, 2026, 11:59 PM</td></tr>
<tr><td><b>Environment</b></td><td>Jupyter Notebook in Visual Studio Code</td></tr>
<tr><td><b>Reference</b></td><td><a href="https://cs231n.github.io/python-numpy-tutorial/">CS231n Python NumPy Tutorial</a></td></tr>
</table>

---

This notebook works through the NumPy portion of the CS231n Python tutorial. Each section has three parts:

1. **Concept.** A short explanation of the idea.
2. **Tutorial examples.** The code from the CS231n tutorial, run and printed.
3. **Going further.** Extra experiments that check how things behave. Many cells end with `assert` statements, so the notebook tests its own claims.

## Table of Contents

| # | Section | Topics |
|---|---------|--------|
| 0 | [Setup](#0.-Setup) | Imports, versions, reproducibility |
| 1 | [NumPy](#1.-NumPy) | What NumPy is and why it is fast |
| 2 | [Arrays](#2.-Arrays) | Rank, shape, constructors, attributes, reshaping |
| 3 | [Array Indexing](#3.-Array-Indexing) | Slicing, integer array indexing, boolean indexing, views vs. copies |
| 4 | [Data Types](#4.-Data-Types) | `dtype`, type inference, casting, overflow, precision |
| 5 | [Array Math](#5.-Array-Math) | Elementwise ops, dot / `@`, reductions, transpose |
| 6 | [Broadcasting](#6.-Broadcasting) | Loops vs. `tile` vs. broadcasting, the rules, applications |
| 7 | [Mini Challenges](#7.-Mini-Challenges) | Vectorized z-score, stable softmax, pairwise distances |
| 8 | [Summary](#8.-Summary) | Key takeaways and references |

### Learning Objectives
- Create and inspect N-dimensional arrays and reason about their **shape** and **rank**.
- Choose the right indexing style (slice, integer array, boolean) and know when the result is a **view** or a **copy**.
- Control numeric **data types** and spot overflow and precision problems.
- Write **vectorized** math with reductions along axes.
- Apply the **broadcasting rules** to replace Python loops with fast array code.

## 0. Setup
We import NumPy, print the environment versions so the results can be reproduced, and seed a random generator so the random outputs are the same on every run.

In [1]:
import sys
import platform
import timeit
import numpy as np

np.set_printoptions(precision=4, suppress=True)   # tidier float output
rng = np.random.default_rng(seed=5831)            # reproducible randomness

print(f"Python   : {sys.version.split()[0]}")
print(f"NumPy    : {np.__version__}")
print(f"Platform : {platform.system()} {platform.machine()}")

Python   : 3.11.15
NumPy    : 2.4.4
Platform : Linux x86_64


## 1. NumPy
**NumPy** is the core library for scientific computing in Python. It provides a fast, memory-efficient **N-dimensional array** object (`ndarray`) and tools for working with it. Libraries such as SciPy, pandas, scikit-learn, PyTorch and TensorFlow use it or copy its design.

**Why it is fast:**
- An `ndarray` stores **one data type** in a **contiguous block of memory**, not a list of pointers to separate Python objects.
- Operations run in **compiled C loops**, which avoids the per-element overhead of the Python interpreter.

The cell below compares a Python list with a NumPy array for the same computation.

In [2]:
n = 1_000_000
py_list = list(range(n))
np_arr  = np.arange(n)

t_list = timeit.timeit(lambda: [x * 2 for x in py_list], number=10) / 10
t_np   = timeit.timeit(lambda: np_arr * 2,               number=10) / 10

print(f"Python list comprehension : {t_list*1e3:8.2f} ms")
print(f"NumPy vectorized          : {t_np*1e3:8.2f} ms")
print(f"Speed-up                  : {t_list / t_np:8.1f}x")

Python list comprehension :    39.16 ms
NumPy vectorized          :     0.83 ms
Speed-up                  :     47.4x


## 2. Arrays
A NumPy array is a grid of values, **all of the same type**, indexed by a tuple of non-negative integers.
- The **rank** (`ndim`) is the number of dimensions.
- The **shape** is a tuple of integers giving the size along each dimension.

### 2.1 Tutorial examples: creating arrays from Python lists

In [3]:
a = np.array([1, 2, 3])   # Create a rank 1 array
print(type(a))            # <class 'numpy.ndarray'>
print(a.shape)            # (3,)
print(a[0], a[1], a[2])   # 1 2 3
a[0] = 5                  # Change an element of the array
print(a)                  # [5 2 3]

b = np.array([[1, 2, 3], [4, 5, 6]])   # Create a rank 2 array
print(b.shape)                          # (2, 3)
print(b[0, 0], b[0, 1], b[1, 0])        # 1 2 4

<class 'numpy.ndarray'>
(3,)
1 2 3
[5 2 3]
(2, 3)
1 2 4


### 2.2 Tutorial examples: built-in constructors

In [4]:
a = np.zeros((2, 2))        # all zeros
print("zeros:\n", a)

b = np.ones((1, 2))         # all ones
print("ones:\n", b)

c = np.full((2, 2), 7)      # constant array
print("full:\n", c)

d = np.eye(2)               # 2x2 identity matrix
print("eye:\n", d)

e = np.random.random((2, 2))  # uniform random values in [0, 1)
print("random:\n", e)

zeros:
 [[0. 0.]
 [0. 0.]]
ones:
 [[1. 1.]]
full:
 [[7 7]
 [7 7]]
eye:
 [[1. 0.]
 [0. 1.]]
random:
 [[0.6267 0.7083]
 [0.9236 0.1179]]


### 2.3 Going further: more constructors and array attributes
`arange` and `linspace` create ranges. Every array also has attributes that describe its layout in memory.

In [5]:
r = np.arange(0, 10, 2)           # start, stop (exclusive), step
l = np.linspace(0, 1, 5)          # 5 evenly spaced points, inclusive
g = rng.normal(0, 1, size=(3, 4)) # Gaussian samples (new Generator API)

def describe(name, x):
    print(f"{name:>2}: ndim={x.ndim} shape={str(x.shape):<7} size={x.size:<3} "
          f"dtype={str(x.dtype):<8} itemsize={x.itemsize}B nbytes={x.nbytes}B")

for name, x in [("r", r), ("l", l), ("g", g)]:
    describe(name, x)

assert g.nbytes == g.size * g.itemsize

 r: ndim=1 shape=(5,)    size=5   dtype=int64    itemsize=8B nbytes=40B
 l: ndim=1 shape=(5,)    size=5   dtype=float64  itemsize=8B nbytes=40B
 g: ndim=2 shape=(3, 4)  size=12  dtype=float64  itemsize=8B nbytes=96B


### 2.4 Going further: reshaping
`reshape` gives the **same data** a different shape. One dimension can be `-1`, and NumPy works out its size. The total number of elements must stay the same.

In [6]:
x = np.arange(12)
print("original :", x.shape)
print("(3, 4):\n", x.reshape(3, 4))
print("(2, -1) ->", x.reshape(2, -1).shape)
print("(2, 3, 2) ->", x.reshape(2, 3, 2).shape)

# Adding an axis is common for broadcasting (see Section 6)
v = np.array([1, 2, 3])
print("row vector   :", v[np.newaxis, :].shape)
print("column vector:", v[:, np.newaxis].shape)

try:
    x.reshape(5, 3)
except ValueError as err:
    print("Expected error ->", err)

original : (12,)
(3, 4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, -1) -> (2, 6)
(2, 3, 2) -> (2, 3, 2)
row vector   : (1, 3)
column vector: (3, 1)
Expected error -> cannot reshape array of size 12 into shape (5,3)


## 3. Array Indexing
NumPy has several ways to index into arrays.

| Style | Syntax example | Result shares memory? |
|-------|----------------|-----------------------|
| Basic slicing | `a[:2, 1:3]` | **Yes** (view) |
| Integer array indexing | `a[[0, 1], [1, 2]]` | No (copy) |
| Boolean indexing | `a[a > 2]` | No (copy) |

### 3.1 Tutorial examples: slicing
As with Python lists, arrays can be sliced. Because arrays can be multidimensional, you give a slice for **each dimension**.

In [7]:
a = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])

# Subarray of the first 2 rows and columns 1 and 2
b = a[:2, 1:3]
print(b)

# A slice is a VIEW into the same data: modifying it modifies the original
print(a[0, 1])   # 2
b[0, 0] = 77     # b[0, 0] is the same data as a[0, 1]
print(a[0, 1])   # 77

[[2 3]
 [6 7]]
2
77


### 3.2 Tutorial examples: mixing integer indexing with slices
An integer index gives an array of **lower rank**. A slice keeps the rank.

In [8]:
a = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])

row_r1 = a[1, :]     # Rank 1 view of the second row
row_r2 = a[1:2, :]   # Rank 2 view of the second row
print(row_r1, row_r1.shape)   # [5 6 7 8] (4,)
print(row_r2, row_r2.shape)   # [[5 6 7 8]] (1, 4)

col_r1 = a[:, 1]
col_r2 = a[:, 1:2]
print(col_r1, col_r1.shape)   # [ 2  6 10] (3,)
print(col_r2, col_r2.shape)   # [[ 2] [ 6] [10]] (3, 1)

[5 6 7 8] (4,)
[[5 6 7 8]] (1, 4)
[ 2  6 10] (3,)
[[ 2]
 [ 6]
 [10]] (3, 1)


### 3.3 Tutorial examples: integer array indexing
Integer array indexing lets you build **arbitrary** arrays from another array's data.

In [9]:
a = np.array([[1, 2], [3, 4], [5, 6]])

print(a[[0, 1, 2], [0, 1, 0]])                  # [1 4 5]
print(np.array([a[0, 0], a[1, 1], a[2, 0]]))    # equivalent

# The same element can be used more than once
print(a[[0, 0], [1, 1]])                        # [2 2]
print(np.array([a[0, 1], a[0, 1]]))             # equivalent

[1 4 5]
[1 4 5]
[2 2]
[2 2]


**A useful trick:** select or change **one element from each row**. The same pattern picks the correct-class score for each sample when computing a classifier's loss.

In [10]:
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
print(a)

b = np.array([0, 2, 0, 1])          # one column index per row
print(a[np.arange(4), b])           # [ 1  6  7 11]

a[np.arange(4), b] += 10            # mutate those elements in place
print(a)

[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]
[ 1  6  7 11]
[[11  2  3]
 [ 4  5 16]
 [17  8  9]
 [10 21 12]]


### 3.4 Tutorial examples: boolean array indexing
Boolean indexing picks out the elements where a condition is true. It is often used to select elements that meet a condition.

In [11]:
a = np.array([[1, 2], [3, 4], [5, 6]])

bool_idx = (a > 2)     # same shape as a; True where a > 2
print(bool_idx)

print(a[bool_idx])     # rank 1 array of the True elements -> [3 4 5 6]
print(a[a > 2])        # same thing in one expression

[[False False]
 [ True  True]
 [ True  True]]
[3 4 5 6]
[3 4 5 6]


### 3.5 Going further: view or copy?
Knowing whether you got a view or a copy prevents bugs where changing one array quietly changes another. `np.shares_memory` answers the question directly.

In [12]:
a = np.arange(12).reshape(3, 4)

cases = {
    "slice      a[:2, 1:3]"      : a[:2, 1:3],
    "reshape    a.reshape(4, 3)" : a.reshape(4, 3),
    "transpose  a.T"             : a.T,
    "int-array  a[[0, 2]]"       : a[[0, 2]],
    "boolean    a[a > 5]"        : a[a > 5],
    "explicit   a.copy()"        : a.copy(),
}
for label, arr in cases.items():
    kind = "VIEW" if np.shares_memory(a, arr) else "COPY"
    print(f"{label:<28} -> {kind}")

slice      a[:2, 1:3]        -> VIEW
reshape    a.reshape(4, 3)   -> VIEW
transpose  a.T               -> VIEW
int-array  a[[0, 2]]         -> COPY
boolean    a[a > 5]          -> COPY
explicit   a.copy()          -> COPY


### 3.6 Going further: conditional replacement with `np.where` and masks

In [13]:
scores = rng.integers(0, 101, size=10)
print("scores     :", scores)

clipped = scores.copy()
clipped[clipped < 50] = 50                       # in-place update through a mask
print("floor at 50:", clipped)

grade = np.where(scores >= 60, "PASS", "FAIL")   # vectorized if/else
print("grade      :", grade)

# Combine conditions with & (and), | (or), ~ (not). Parentheses are required.
print("40-80 range:", scores[(scores >= 40) & (scores <= 80)])

scores     : [10 96 67 96 61 91 81 13 83 63]
floor at 50: [50 96 67 96 61 91 81 50 83 63]
grade      : ['FAIL' 'PASS' 'PASS' 'PASS' 'PASS' 'PASS' 'PASS' 'FAIL' 'PASS' 'PASS']
40-80 range: [67 61 63]


## 4. Data Types
Every NumPy array is a grid of elements of the **same type**. NumPy guesses a type when you create an array. You can also set it yourself with the `dtype` argument.

### 4.1 Tutorial examples

In [14]:
x = np.array([1, 2])                   # NumPy chooses the datatype
print(x.dtype)                         # int64

x = np.array([1.0, 2.0])               # NumPy chooses the datatype
print(x.dtype)                         # float64

x = np.array([1, 2], dtype=np.int64)   # force a specific datatype
print(x.dtype)                         # int64

int64
float64
int64


### 4.2 Going further: type inference, casting and upcasting
- When types are mixed, NumPy **upcasts** to the most general type (`bool` → `int` → `float` → `complex`).
- `astype` always returns a **new** array. Converting float to int **truncates** toward zero; it does not round.
- `np.round` uses **round-half-to-even** ("banker's rounding"), so `2.5` becomes `2`.

In [15]:
print(np.array([1, 2.5]).dtype)          # int + float    -> float64
print(np.array([True, 2]).dtype)         # bool + int     -> int64
print(np.array([1, 2 + 3j]).dtype)       # int + complex  -> complex128
print(np.array([1, "a"]).dtype)          # mixed with str -> Unicode string

f = np.array([1.7, -1.7, 2.5])
print("astype(int)      :", f.astype(np.int32))   # truncation
print("round then astype:", np.round(f).astype(np.int32))   # note: 2.5 -> 2 (round-half-to-even)

float64
int64
complex128
<U21
astype(int)      : [ 1 -1  2]
round then astype: [ 2 -2  2]


### 4.3 Going further: fixed-width integers can overflow
Unlike Python's arbitrary-precision `int`, NumPy integers have a **fixed width**. Going past the limit **wraps around** without an error. This is important for image data stored as `uint8`.

In [16]:
for t in (np.int8, np.uint8, np.int32, np.int64):
    info = np.iinfo(t)
    print(f"{t.__name__:<7} range: [{info.min}, {info.max}]")

pixels = np.array([250, 100], dtype=np.uint8)
print("\nuint8 250 + 10 ->", pixels + np.uint8(10))       # wraps to 4
print("safe (upcast)  ->", pixels.astype(np.int16) + 10)  # 260 as expected

int8    range: [-128, 127]
uint8   range: [0, 255]
int32   range: [-2147483648, 2147483647]
int64   range: [-9223372036854775808, 9223372036854775807]

uint8 250 + 10 -> [  4 110]
safe (upcast)  -> [260 110]


### 4.4 Going further: floating-point precision and memory
Lower precision saves memory, which matters in deep learning, but costs accuracy. Compare floats with `np.isclose`, not `==`.

In [17]:
for t in (np.float16, np.float32, np.float64):
    fi = np.finfo(t)
    print(f"{t.__name__:<8} eps={fi.eps:<10.3e} ~decimal digits={fi.precision:<3} "
          f"1M elements = {np.zeros(1_000_000, dtype=t).nbytes/1e6:.0f} MB")

print("\n0.1 + 0.2 == 0.3          ->", 0.1 + 0.2 == 0.3)
print("np.isclose(0.1 + 0.2, 0.3) ->", np.isclose(0.1 + 0.2, 0.3))

float16  eps=9.766e-04  ~decimal digits=3   1M elements = 2 MB
float32  eps=1.192e-07  ~decimal digits=6   1M elements = 4 MB
float64  eps=2.220e-16  ~decimal digits=15  1M elements = 8 MB

0.1 + 0.2 == 0.3          -> False
np.isclose(0.1 + 0.2, 0.3) -> True


## 5. Array Math
Basic math functions work **elementwise** on arrays. Each is available both as an operator and as a NumPy function.

### 5.1 Tutorial examples: elementwise operations

In [18]:
x = np.array([[1, 2], [3, 4]], dtype=np.float64)
y = np.array([[5, 6], [7, 8]], dtype=np.float64)

print("x + y:\n", x + y);  print(np.add(x, y))
print("x - y:\n", x - y);  print(np.subtract(x, y))
print("x * y:\n", x * y);  print(np.multiply(x, y))
print("x / y:\n", x / y);  print(np.divide(x, y))
print("sqrt(x):\n", np.sqrt(x))

x + y:
 [[ 6.  8.]
 [10. 12.]]
[[ 6.  8.]
 [10. 12.]]
x - y:
 [[-4. -4.]
 [-4. -4.]]
[[-4. -4.]
 [-4. -4.]]
x * y:
 [[ 5. 12.]
 [21. 32.]]
[[ 5. 12.]
 [21. 32.]]
x / y:
 [[0.2    0.3333]
 [0.4286 0.5   ]]
[[0.2    0.3333]
 [0.4286 0.5   ]]
sqrt(x):
 [[1.     1.4142]
 [1.7321 2.    ]]


### 5.2 Tutorial examples: dot products and matrix multiplication
Note that `*` is **elementwise** multiplication, not matrix multiplication. Use `dot` (or the `@` operator) to compute inner products, multiply a vector by a matrix, and multiply matrices.

In [19]:
x = np.array([[1, 2], [3, 4]])
y = np.array([[5, 6], [7, 8]])
v = np.array([9, 10])
w = np.array([11, 12])

# Inner product of vectors -> 219
print(v.dot(w)); print(np.dot(v, w)); print(v @ w)

# Matrix / vector product -> rank 1 array [29 67]
print(x.dot(v)); print(np.dot(x, v)); print(x @ v)

# Matrix / matrix product -> rank 2 array
print(x.dot(y)); print(np.dot(x, y)); print(x @ y)

219
219
219
[29 67]
[29 67]
[29 67]
[[19 22]
 [43 50]]
[[19 22]
 [43 50]]
[[19 22]
 [43 50]]


### 5.3 Tutorial examples: reductions (`sum`) along an axis
`axis=0` reduces **down the rows**, giving one result per column. `axis=1` reduces **across the columns**, giving one result per row.

In [20]:
x = np.array([[1, 2], [3, 4]])

print(np.sum(x))           # sum of all elements          -> 10
print(np.sum(x, axis=0))   # sum of each column           -> [4 6]
print(np.sum(x, axis=1))   # sum of each row              -> [3 7]

10
[4 6]
[3 7]


### 5.4 Tutorial examples: transpose

In [21]:
x = np.array([[1, 2], [3, 4]])
print(x)
print(x.T)                 # [[1 3] [2 4]]

v = np.array([1, 2, 3])
print(v)                   # transposing a rank 1 array does nothing
print(v.T)                 # [1 2 3]

[[1 2]
 [3 4]]
[[1 3]
 [2 4]]
[1 2 3]
[1 2 3]


### 5.5 Going further: more reductions, `keepdims`, and checking identities

In [22]:
M = rng.integers(1, 10, size=(3, 4))
print("M:\n", M)
print("mean per column :", M.mean(axis=0))
print("max per row     :", M.max(axis=1))
print("argmax per row  :", M.argmax(axis=1))
print("std (all)       :", round(M.std(), 4))

# keepdims keeps the reduced axis with size 1, so the result broadcasts back against M
row_sums = M.sum(axis=1, keepdims=True)
print("row_sums shape  :", row_sums.shape)
print("row-normalized (each row sums to 1):\n", M / row_sums)

# Check linear-algebra identities numerically
A = rng.normal(size=(3, 4)); B = rng.normal(size=(4, 2))
assert np.allclose((A @ B).T, B.T @ A.T)            # (AB)^T = B^T A^T
assert np.allclose(np.sum(M, axis=0).sum(), M.sum())
print("identities hold")

M:
 [[7 8 6 4]
 [2 5 3 2]
 [4 5 8 1]]
mean per column : [4.3333 6.     5.6667 2.3333]
max per row     : [8 5 8]
argmax per row  : [1 1 2]
std (all)       : 2.2531
row_sums shape  : (3, 1)
row-normalized (each row sums to 1):
 [[0.28   0.32   0.24   0.16  ]
 [0.1667 0.4167 0.25   0.1667]
 [0.2222 0.2778 0.4444 0.0556]]
identities hold


## 6. Broadcasting
Broadcasting lets NumPy work with arrays of **different shapes** in arithmetic. It is useful when we have a small array and a large array and want to apply the small one to the large one many times.

**Task:** add the constant vector `v` to every row of matrix `x`. The tutorial solves this three ways.

### 6.1 Approach 1: explicit Python loop

In [23]:
x = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
v = np.array([1, 0, 1])
y = np.empty_like(x)

for i in range(4):
    y[i, :] = x[i, :] + v
print(y)

[[ 2  2  4]
 [ 5  5  7]
 [ 8  8 10]
 [11 11 13]]


### 6.2 Approach 2: stack copies of `v` with `np.tile`

In [24]:
vv = np.tile(v, (4, 1))   # stack 4 copies of v on top of each other
print(vv)
y = x + vv
print(y)

[[1 0 1]
 [1 0 1]
 [1 0 1]
 [1 0 1]]
[[ 2  2  4]
 [ 5  5  7]
 [ 8  8 10]
 [11 11 13]]


### 6.3 Approach 3: broadcasting
NumPy repeats `v` along the missing dimension **without copying any data**.

In [25]:
y = x + v                 # v (3,) broadcasts against x (4, 3)
print(y)

[[ 2  2  4]
 [ 5  5  7]
 [ 8  8 10]
 [11 11 13]]


### 6.4 The broadcasting rules
To combine two arrays, NumPy compares their shapes **starting from the last dimension**:

1. If the arrays have different ranks, **prepend 1s** to the shape of the lower-rank array until the ranks match.
2. Two dimensions are **compatible** if they are equal or if one of them is **1**.
3. In any dimension where one array has size 1 and the other is larger, the size-1 array acts as if it were **copied** along that dimension.
4. If any dimension is incompatible, NumPy raises an error.

`np.broadcast_shapes` lets us test the rules without creating any arrays:

In [26]:
tests = [((4, 3), (3,)), ((3, 1), (1, 2)), ((2, 3), (2, 1)),
         ((8, 1, 6, 1), (7, 1, 5)), ((2, 3), (3, 2))]

for s1, s2 in tests:
    try:
        print(f"{str(s1):<14} + {str(s2):<10} -> {np.broadcast_shapes(s1, s2)}")
    except ValueError:
        print(f"{str(s1):<14} + {str(s2):<10} -> incompatible")

(4, 3)         + (3,)       -> (4, 3)
(3, 1)         + (1, 2)     -> (3, 2)
(2, 3)         + (2, 1)     -> (2, 3)
(8, 1, 6, 1)   + (7, 1, 5)  -> (8, 7, 6, 5)
(2, 3)         + (3, 2)     -> incompatible


### 6.5 Tutorial examples: applications of broadcasting

In [27]:
v = np.array([1, 2, 3])   # shape (3,)
w = np.array([4, 5])      # shape (2,)
x = np.array([[1, 2, 3], [4, 5, 6]])

# (1) Outer product: reshape v to a (3, 1) column, then broadcast against w (2,) -> (3, 2)
print(np.reshape(v, (3, 1)) * w)

# (2) Add a vector to each row: x (2, 3) + v (3,) -> (2, 3)
print(x + v)

# (3) Add a vector to each column: x (2, 3), w (2,)
#     Option A: transpose, broadcast, transpose back
print((x.T + w).T)
#     Option B: reshape w into a (2, 1) column
print(x + np.reshape(w, (2, 1)))

# (4) Multiply by a scalar: the scalar broadcasts to shape ()
print(x * 2)

[[ 4  5]
 [ 8 10]
 [12 15]]
[[2 4 6]
 [5 7 9]]
[[ 5  6  7]
 [ 9 10 11]]
[[ 5  6  7]
 [ 9 10 11]]
[[ 2  4  6]
 [ 8 10 12]]


### 6.6 Going further: benchmark (loop vs. tile vs. broadcasting)
We repeat the Section 6.1–6.3 task on a larger matrix and time all three approaches. Broadcasting is the fastest, and unlike `tile` it does not allocate a temporary copy of `v`.

In [28]:
X = rng.normal(size=(20_000, 64))
v = rng.normal(size=64)

def loop_add(X, v):
    Y = np.empty_like(X)
    for i in range(X.shape[0]):
        Y[i, :] = X[i, :] + v
    return Y

def tile_add(X, v):
    return X + np.tile(v, (X.shape[0], 1))

def broadcast_add(X, v):
    return X + v

assert np.allclose(loop_add(X, v), broadcast_add(X, v))
assert np.allclose(tile_add(X, v), broadcast_add(X, v))

results = {}
for fn in (loop_add, tile_add, broadcast_add):
    results[fn.__name__] = timeit.timeit(lambda: fn(X, v), number=5) / 5

base = results["loop_add"]
for name, t in results.items():
    print(f"{name:<14}: {t*1e3:8.2f} ms   ({base / t:6.1f}x vs loop)")

loop_add      :    13.04 ms   (   1.0x vs loop)
tile_add      :     3.88 ms   (   3.4x vs loop)
broadcast_add :     1.43 ms   (   9.1x vs loop)


## 7. Mini Challenges
Three short problems that use everything above. Each solution is **fully vectorized** (no Python loops over data) and is checked against a reference with `assert`.

### 7.1 Feature standardization (z-score)
Given a data matrix $X \in \mathbb{R}^{N \times D}$, standardize each feature (column) to zero mean and unit variance:
$$\hat{X}_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}$$

In [29]:
X = rng.normal(loc=[10, -3, 100], scale=[2, 0.5, 25], size=(500, 3))

mu    = X.mean(axis=0)          # shape (3,)
sigma = X.std(axis=0)           # shape (3,)
X_hat = (X - mu) / sigma        # (500, 3) op (3,) -> broadcasting

print("means after :", X_hat.mean(axis=0))
print("stds after  :", X_hat.std(axis=0))
assert np.allclose(X_hat.mean(axis=0), 0) and np.allclose(X_hat.std(axis=0), 1)

means after : [ 0. -0.  0.]
stds after  : [1. 1. 1.]


### 7.2 Numerically stable softmax (row-wise)
$$\text{softmax}(z)_k = \frac{e^{z_k - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$
Subtracting the row maximum does not change the result but prevents `exp` from overflowing. This uses `keepdims=True` and broadcasting.

In [30]:
def softmax(Z):
    Z_shift = Z - Z.max(axis=1, keepdims=True)   # (N, C) - (N, 1)
    E = np.exp(Z_shift)
    return E / E.sum(axis=1, keepdims=True)

logits = np.array([[1.0, 2.0, 3.0],
                   [1000.0, 1001.0, 1002.0]])   # naive exp(1000) would overflow

P = softmax(logits)
print(P)
assert np.allclose(P.sum(axis=1), 1)
assert np.allclose(P[0], P[1])                  # softmax is shift-invariant

[[0.09   0.2447 0.6652]
 [0.09   0.2447 0.6652]]


### 7.3 Pairwise Euclidean distances without loops
For points $A \in \mathbb{R}^{M \times D}$ and $B \in \mathbb{R}^{N \times D}$, compute $D_{ij} = \lVert a_i - b_j \rVert_2$.

Broadcasting `A[:, None, :] - B[None, :, :]` gives an `(M, N, D)` array of differences. This is the same idea used in the k-nearest-neighbor assignment in CS231n.

In [31]:
A = rng.normal(size=(5, 3))
B = rng.normal(size=(4, 3))

# Vectorized: (5,1,3) - (1,4,3) -> (5,4,3) -> reduce over D
D_vec = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1))

# Reference: double loop
D_loop = np.zeros((5, 4))
for i in range(5):
    for j in range(4):
        D_loop[i, j] = np.sqrt(np.sum((A[i] - B[j]) ** 2))

print(D_vec)
assert D_vec.shape == (5, 4)
assert np.allclose(D_vec, D_loop)
print("vectorized result matches the loop reference")

[[1.6056 2.0128 2.199  2.1242]
 [1.4182 0.5126 1.8186 1.3949]
 [2.7869 2.6225 2.9893 1.3836]
 [1.5252 0.9778 0.9138 2.2702]
 [5.2813 4.4677 4.1836 3.6701]]
vectorized result matches the loop reference


## 8. Summary

| Topic | Key takeaway |
|-------|--------------|
| **NumPy** | Contiguous, single-type arrays and compiled loops make vectorized code much faster than pure Python. |
| **Arrays** | Understand `shape` and `ndim` first. Use `zeros`, `ones`, `full`, `eye`, `arange`, `linspace` to build arrays and `reshape` / `np.newaxis` to change their shape. |
| **Array indexing** | Slices return **views**. Integer array and boolean indexing return **copies**. `a[np.arange(N), idx]` selects one element per row. |
| **Data types** | NumPy infers and upcasts types. Fixed-width integers **wrap on overflow**. Compare floats with `np.isclose`. |
| **Array math** | `*` is elementwise and `@` / `dot` is matrix multiplication. Reductions take an `axis`, and `keepdims` keeps results ready to broadcast. |
| **Broadcasting** | Shapes are aligned from the right. Dimensions must be equal or 1. It replaces loops and `tile` without copying data. |

### References
1. Justin Johnson, Volodymyr Kuleshov, Isaac Caswell. *CS231n Python NumPy Tutorial.* https://cs231n.github.io/python-numpy-tutorial/
2. NumPy Documentation. *Broadcasting.* https://numpy.org/doc/stable/user/basics.broadcasting.html
3. NumPy Documentation. *Indexing on ndarrays.* https://numpy.org/doc/stable/user/basics.indexing.html
4. NumPy Documentation. *Data types.* https://numpy.org/doc/stable/user/basics.types.html

---
*Nabonita Das · ECE 5831 · HW#2 (A_02) · Completed in Jupyter Notebook in VS Code*